# 机器学习基础指南

本 notebook 覆盖机器学习的核心基础，不涉及深度学习方向。

**学习路径：**
1. 什么是机器学习
2. ML 的三大类型
3. 核心概念与术语
4. 监督学习算法
5. 无监督学习算法
6. 模型评估与选择
7. 特征工程基础
8. 超参数调优
9. 完整 ML 工作流实战
10. 总结与学习建议

In [ ]:
# !pip install numpy pandas matplotlib seaborn scikit-learn

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn import datasets

plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
sns.set_theme(style='whitegrid')
print('环境初始化完成')

---
## 第一章：什么是机器学习

**传统编程 vs 机器学习：**

```
传统编程：数据 + 规则  →  答案
机器学习：数据 + 答案  →  规则（模型）
```

**定义：** 让计算机从数据中自动学习规律，而不是通过显式编程来完成任务。

**核心要素：**
- **数据（Data）**：学习的原材料
- **模型（Model）**：从数据中学到的规律的数学表示
- **损失函数（Loss Function）**：衡量模型预测与真实答案的差距
- **优化（Optimization）**：通过最小化损失来改进模型

---
## 第二章：ML 的三大类型

### 2.1 监督学习（Supervised Learning）
- **特征**：训练数据有标签（输入 + 正确答案）
- **任务**：回归（预测连续值）、分类（预测类别）
- **例子**：房价预测、垃圾邮件过滤、图像分类

### 2.2 无监督学习（Unsupervised Learning）
- **特征**：训练数据无标签（只有输入）
- **任务**：聚类（发现分组）、降维（压缩特征）
- **例子**：客户分群、异常检测、数据压缩

### 2.3 强化学习（Reinforcement Learning）
- **特征**：智能体通过与环境交互获得奖励信号来学习
- **例子**：游戏 AI、机器人控制（本指南不深入此方向）

In [ ]:
# 可视化三大类型
from sklearn.cluster import KMeans
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

np.random.seed(42)
c1 = np.random.randn(30, 2) + [2, 2]
c2 = np.random.randn(30, 2) + [-2, -2]
axes[0].scatter(c1[:,0], c1[:,1], c='steelblue', label='类别 A', alpha=0.7)
axes[0].scatter(c2[:,0], c2[:,1], c='coral',     label='类别 B', alpha=0.7)
axes[0].axline((0,0), slope=1, color='gray', linestyle='--', lw=1.5)
axes[0].set_title('监督学习（分类）', fontsize=13)
axes[0].legend()

data = np.vstack([
    np.random.randn(40, 2) + [3, 3],
    np.random.randn(40, 2) + [-3, 0],
    np.random.randn(40, 2) + [0, -3]
])
labels = KMeans(n_clusters=3, random_state=42, n_init=10).fit_predict(data)
for i, col in enumerate(['steelblue', 'coral', 'seagreen']):
    axes[1].scatter(data[labels==i,0], data[labels==i,1], c=col, label=f'簇 {i+1}', alpha=0.7)
axes[1].set_title('无监督学习（聚类）', fontsize=13)
axes[1].legend()

x = np.linspace(0, 10, 50)
y = 2 * x + 1 + np.random.randn(50) * 2
axes[2].scatter(x, y, alpha=0.6, color='steelblue', label='数据点')
axes[2].plot(x, 2*x+1, 'r-', lw=2, label='拟合直线')
axes[2].set_title('监督学习（回归）', fontsize=13)
axes[2].legend()

plt.tight_layout()
plt.show()

---
## 第三章：核心概念与术语

| 术语 | 解释 | 例子 |
|------|------|------|
| **特征（Feature）** | 输入变量，描述样本的属性 | 房屋面积、卧室数量 |
| **标签（Label）** | 目标变量，要预测的值 | 房价 |
| **样本（Sample）** | 一条数据记录 | 一套房子的信息 |
| **训练集** | 用于训练模型的数据 | 80% 数据 |
| **测试集** | 用于最终评估模型的数据 | 20% 数据 |
| **验证集** | 调参时评估模型的数据 | 从训练集中划出 |
| **超参数** | 人为设定的参数（非学习得到）| 决策树深度 |
| **过拟合** | 模型对训练集太好，泛化差 | 只会背题答案 |
| **欠拟合** | 模型没学到数据规律 | 学习不够充分 |

In [ ]:
from sklearn.model_selection import train_test_split

iris = datasets.load_iris()
X, y = iris.data, iris.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'总样本数：{len(X)}')
print(f'训练集：{len(X_train)} 条 ({len(X_train)/len(X):.0%})')
print(f'测试集：{len(X_test)} 条  ({len(X_test)/len(X):.0%})')
print(f'特征维度：{X.shape[1]}（花萼长/宽、花瓣长/宽）')
print(f'类别：{iris.target_names}')

---
## 第四章：监督学习算法

### 4.1 线性回归（Linear Regression）

**原理：** 用一条直线（或超平面）拟合数据

$$\hat{y} = w_0 + w_1 x_1 + w_2 x_2 + \cdots + w_n x_n$$

**损失函数（MSE）：**

$$\text{MSE} = \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2$$

**适用场景：** 预测连续值，特征与目标呈线性关系

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

np.random.seed(0)
X_reg = np.random.rand(100, 1) * 10
y_reg = 3 * X_reg.ravel() + 5 + np.random.randn(100) * 3

X_tr, X_te, y_tr, y_te = train_test_split(X_reg, y_reg, test_size=0.2, random_state=0)

model = LinearRegression()
model.fit(X_tr, y_tr)
y_pred = model.predict(X_te)

print(f'截距 w0 = {model.intercept_:.2f}')
print(f'斜率 w1 = {model.coef_[0]:.2f}（真实值为 3）')
print(f'MSE  = {mean_squared_error(y_te, y_pred):.2f}')
print(f'R2   = {r2_score(y_te, y_pred):.3f}  （越接近 1 越好）')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
x_line = np.linspace(0, 10, 100).reshape(-1, 1)
axes[0].scatter(X_tr, y_tr, alpha=0.5, label='训练点')
axes[0].scatter(X_te, y_te, alpha=0.8, marker='*', s=80, label='测试点')
axes[0].plot(x_line, model.predict(x_line), 'r-', lw=2, label='拟合线')
axes[0].set_title('线性回归拟合')
axes[0].legend()

residuals = y_te - y_pred
axes[1].scatter(y_pred, residuals, alpha=0.7)
axes[1].axhline(0, color='r', linestyle='--')
axes[1].set_xlabel('预测值')
axes[1].set_ylabel('残差')
axes[1].set_title('残差图（理想情况应随机分布在 0 周围）')
plt.tight_layout()
plt.show()

### 4.2 逻辑回归（Logistic Regression）

**原理：** 在线性回归基础上加 Sigmoid 函数，输出概率（0~1），用于分类

$$\hat{p} = \sigma(z) = \frac{1}{1 + e^{-z}}$$

**损失函数：** 二元交叉熵（Binary Cross-Entropy）

**适用场景：** 二分类问题（也可扩展到多分类）

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

mask = y < 2
X_bin, y_bin = X[mask, :2], y[mask]

X_tr, X_te, y_tr, y_te = train_test_split(X_bin, y_bin, test_size=0.2, random_state=42)

clf = LogisticRegression(random_state=42)
clf.fit(X_tr, y_tr)
y_pred = clf.predict(X_te)

print(f'准确率：{accuracy_score(y_te, y_pred):.2%}')
print()
print(classification_report(y_te, y_pred, target_names=iris.target_names[:2]))

def plot_boundary(clf, X, y, ax, title):
    h = 0.02
    x0_min, x0_max = X[:,0].min()-0.5, X[:,0].max()+0.5
    x1_min, x1_max = X[:,1].min()-0.5, X[:,1].max()+0.5
    xx, yy = np.meshgrid(np.arange(x0_min, x0_max, h), np.arange(x1_min, x1_max, h))
    Z = clf.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.3, cmap='coolwarm')
    ax.scatter(X[:,0], X[:,1], c=y, cmap='coolwarm', edgecolors='k', s=40)
    ax.set_title(title)
    ax.set_xlabel(iris.feature_names[0])
    ax.set_ylabel(iris.feature_names[1])

fig, ax = plt.subplots(figsize=(7, 5))
plot_boundary(clf, X_bin, y_bin, ax, '逻辑回归决策边界')
plt.tight_layout()
plt.show()

### 4.3 决策树（Decision Tree）

**原理：** 通过一系列 if-else 判断规则将数据划分，每个节点代表一个判断条件

**分裂标准：**
- 分类：基尼系数（Gini）、信息增益（Entropy）
- 回归：均方误差（MSE）

**优点：** 可解释性强，无需特征标准化  
**缺点：** 容易过拟合，需要剪枝（控制 max_depth）

In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

dt = DecisionTreeClassifier(max_depth=3, random_state=42)
dt.fit(X_tr, y_tr)

print(f'训练集准确率：{dt.score(X_tr, y_tr):.2%}')
print(f'测试集准确率：{dt.score(X_te, y_te):.2%}')

fig, ax = plt.subplots(figsize=(16, 6))
plot_tree(dt, feature_names=iris.feature_names,
          class_names=iris.target_names,
          filled=True, fontsize=10, ax=ax)
ax.set_title('决策树结构（max_depth=3）', fontsize=14)
plt.tight_layout()
plt.show()

### 4.4 K 近邻（K-Nearest Neighbors, KNN）

**原理：** 预测时找训练集中最近的 K 个邻居，投票决定类别（物以类聚）

**距离度量：** 常用欧氏距离 $d = \sqrt{\sum (x_i - x_j)^2}$

**超参数 K 的选择：**
- K 太小 → 过拟合（噪声敏感）
- K 太大 → 欠拟合（边界模糊）
- 通常用奇数，通过交叉验证选择

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_tr, X_te, y_tr, y_te = train_test_split(X_scaled, y, test_size=0.2, random_state=42, stratify=y)

k_values = range(1, 21)
train_scores, test_scores = [], []
for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_tr, y_tr)
    train_scores.append(knn.score(X_tr, y_tr))
    test_scores.append(knn.score(X_te, y_te))

best_k = list(k_values)[np.argmax(test_scores)]
print(f'最优 K = {best_k}，测试集准确率 = {max(test_scores):.2%}')

plt.figure(figsize=(9, 4))
plt.plot(k_values, train_scores, 'o-', label='训练集')
plt.plot(k_values, test_scores,  's-', label='测试集')
plt.axvline(best_k, color='r', linestyle='--', alpha=0.7, label=f'最优 K={best_k}')
plt.xlabel('K 值')
plt.ylabel('准确率')
plt.title('K 值对 KNN 性能的影响')
plt.legend()
plt.tight_layout()
plt.show()

### 4.5 支持向量机（SVM）

**原理：** 找到间隔最大的决策超平面（最大化支持向量之间的边距）

**核函数（Kernel）：** 通过映射处理非线性问题
- `linear`：线性核
- `rbf`：径向基核（最常用）
- `poly`：多项式核

**关键超参数：**
- `C`：惩罚系数，越大模型越严格
- `gamma`：rbf 核的影响范围

In [ ]:
from sklearn.svm import SVC

X_tr, X_te, y_tr, y_te = train_test_split(X_scaled, y, test_size=0.2, random_state=42, stratify=y)

print(f'{"核函数":<10} {"测试准确率"}')
print('-' * 22)
for kernel in ['linear', 'rbf', 'poly']:
    svm = SVC(kernel=kernel, random_state=42)
    svm.fit(X_tr, y_tr)
    print(f'{kernel:<12} {svm.score(X_te, y_te):.2%}')

### 4.6 随机森林（Random Forest）

**原理：** 集成方法——训练多棵决策树，通过投票/平均来提升性能

**两种随机性：**
1. **Bootstrap 采样**：每棵树使用不同的随机子集
2. **特征随机**：每次分裂只考虑随机子集的特征

**优点：** 抗过拟合、可估计特征重要性、效果好

In [ ]:
from sklearn.ensemble import RandomForestClassifier

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_tr, y_tr)
print(f'随机森林测试准确率：{rf.score(X_te, y_te):.2%}')

importances = rf.feature_importances_
indices = np.argsort(importances)[::-1]
feat_names = ['萼长', '萼宽', '瓣长', '瓣宽']

plt.figure(figsize=(7, 4))
plt.bar(range(4), importances[indices], color='steelblue')
plt.xticks(range(4), [feat_names[i] for i in indices])
plt.title('随机森林特征重要性')
plt.ylabel('重要性分数')
plt.tight_layout()
plt.show()

---
## 第五章：无监督学习算法

### 5.1 K-Means 聚类

**算法步骤：**
1. 随机初始化 K 个中心点
2. 将每个点分配给最近的中心
3. 重新计算每个簇的中心
4. 重复 2-3 直到收敛

**确定 K 的方法：肘部法（Elbow Method）**

In [ ]:
from sklearn.metrics import silhouette_score

X_scaled_all = StandardScaler().fit_transform(X)

inertias, silhouettes = [], []
K_range = range(2, 9)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    lbl = km.fit_predict(X_scaled_all)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled_all, lbl))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(K_range, inertias, 'o-')
axes[0].axvline(3, color='r', linestyle='--', alpha=0.7, label='K=3（真实类别数）')
axes[0].set_xlabel('K')
axes[0].set_ylabel('Inertia（簇内误差）')
axes[0].set_title('肘部法（拐点处为最优 K）')
axes[0].legend()

axes[1].plot(K_range, silhouettes, 's-', color='coral')
axes[1].axvline(3, color='r', linestyle='--', alpha=0.7, label='K=3')
axes[1].set_xlabel('K')
axes[1].set_ylabel('轮廓系数（越大越好）')
axes[1].set_title('轮廓系数法')
axes[1].legend()

plt.tight_layout()
plt.show()

### 5.2 主成分分析（PCA）

**目的：** 降维——在保留最多信息的前提下，将高维数据映射到低维空间

**原理：** 找到方差最大的方向（主成分）作为新的坐标轴

**用途：** 可视化高维数据、去除冗余特征、去噪

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled_all)

print('各主成分方差解释比例：')
for i, r in enumerate(pca.explained_variance_ratio_):
    print(f'  PC{i+1}: {r:.2%}')
print(f'  累计：{pca.explained_variance_ratio_.sum():.2%}')

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for label, name in enumerate(iris.target_names):
    mask = y == label
    axes[0].scatter(X_pca[mask,0], X_pca[mask,1], label=name, alpha=0.8, s=50)
axes[0].set_xlabel('PC1')
axes[0].set_ylabel('PC2')
axes[0].set_title('PCA 降维（按真实标签）')
axes[0].legend()

km3 = KMeans(n_clusters=3, random_state=42, n_init=10)
cluster_labels = km3.fit_predict(X_scaled_all)
for c in range(3):
    mask = cluster_labels == c
    axes[1].scatter(X_pca[mask,0], X_pca[mask,1], label=f'簇 {c+1}', alpha=0.8, s=50)
axes[1].set_xlabel('PC1')
axes[1].set_ylabel('PC2')
axes[1].set_title('PCA 降维（按 K-Means 聚类）')
axes[1].legend()

plt.tight_layout()
plt.show()

---
## 第六章：模型评估与选择

### 6.1 分类评估指标

**混淆矩阵（Confusion Matrix）：**

```
              预测正       预测负
实际正   TP（真正例）  FN（漏报）
实际负   FP（误报）   TN（真负例）
```

| 指标 | 公式 | 说明 |
|------|------|------|
| 准确率（Accuracy） | (TP+TN)/All | 总体正确率 |
| 精确率（Precision） | TP/(TP+FP) | 预测正例中真正正例的比例 |
| 召回率（Recall） | TP/(TP+FN) | 真正正例被找到的比例 |
| F1 Score | 2*P*R/(P+R) | 精确率与召回率的调和平均 |

**ROC 曲线 & AUC：** 衡量模型区分正负例的能力（AUC 越接近 1 越好）

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, roc_curve, auc
from sklearn.preprocessing import label_binarize

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_tr, y_tr)
y_pred = rf.predict(X_te)
y_prob = rf.predict_proba(X_te)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

cm = confusion_matrix(y_te, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=iris.target_names)
disp.plot(ax=axes[0], colorbar=False)
axes[0].set_title('混淆矩阵')

y_bin = label_binarize(y_te, classes=[0, 1, 2])
for i, name in enumerate(iris.target_names):
    fpr, tpr, _ = roc_curve(y_bin[:, i], y_prob[:, i])
    axes[1].plot(fpr, tpr, label=f'{name} (AUC={auc(fpr,tpr):.2f})')
axes[1].plot([0,1],[0,1],'k--', alpha=0.5, label='随机猜测')
axes[1].set_xlabel('假阳率 (FPR)')
axes[1].set_ylabel('真阳率 (TPR)')
axes[1].set_title('ROC 曲线')
axes[1].legend()

plt.tight_layout()
plt.show()

### 6.2 交叉验证（Cross-Validation）

**K 折交叉验证：** 将数据分成 K 份，轮流用其中一份做测试，其余做训练，取平均值

解决单次划分结果不稳定的问题。

In [ ]:
from sklearn.model_selection import cross_val_score, KFold

models = {
    '逻辑回归': LogisticRegression(max_iter=200, random_state=42),
    '决策树':   DecisionTreeClassifier(max_depth=3, random_state=42),
    'KNN':      KNeighborsClassifier(n_neighbors=5),
    'SVM':      SVC(kernel='rbf', random_state=42),
    '随机森林': RandomForestClassifier(n_estimators=100, random_state=42),
}

X_s = StandardScaler().fit_transform(X)
kf = KFold(n_splits=5, shuffle=True, random_state=42)

print(f'{"模型":<10} {"均值":>8} {"标准差":>8}')
print('-' * 28)
results = {}
for name, m in models.items():
    scores = cross_val_score(m, X_s, y, cv=kf, scoring='accuracy')
    results[name] = scores
    print(f'{name:<10} {scores.mean():>7.2%} {scores.std():>8.2%}')

fig, ax = plt.subplots(figsize=(9, 4))
ax.boxplot(results.values(), labels=results.keys())
ax.set_ylabel('5 折交叉验证准确率')
ax.set_title('各算法性能对比（5-Fold CV）')
ax.set_ylim(0.8, 1.02)
plt.tight_layout()
plt.show()

### 6.3 过拟合与欠拟合

```
            欠拟合        合适拟合       过拟合
偏差(Bias)    高             低            低
方差(Variance)低             低            高
训练集        差             好           很好
测试集        差             好            差
```

**解决方案：**
- 过拟合：增加数据量、正则化、减小模型复杂度
- 欠拟合：增加模型复杂度、增加特征、减少正则化

In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

depths = range(1, 15)
train_acc, test_acc = [], []
for d in depths:
    dt = DecisionTreeClassifier(max_depth=d, random_state=42)
    dt.fit(X_tr, y_tr)
    train_acc.append(dt.score(X_tr, y_tr))
    test_acc.append(dt.score(X_te, y_te))

plt.figure(figsize=(9, 4))
plt.plot(depths, train_acc, 'o-', label='训练集')
plt.plot(depths, test_acc,  's-', label='测试集')
plt.axvspan(1, 2.5, alpha=0.1, color='blue', label='欠拟合区域')
plt.axvspan(7, 14,  alpha=0.1, color='red',  label='过拟合区域')
plt.xlabel('树的最大深度 (max_depth)')
plt.ylabel('准确率')
plt.title('偏差-方差权衡：模型复杂度 vs 准确率')
plt.legend()
plt.tight_layout()
plt.show()

---
## 第七章：特征工程基础

### 7.1 特征缩放

| 方法 | 公式 | 适用场景 |
|------|------|----------|
| 标准化（StandardScaler） | $(x - \mu) / \sigma$ | KNN、SVM、线性模型、PCA |
| 归一化（MinMaxScaler） | $(x - x_{min}) / (x_{max} - x_{min})$ | 神经网络、图像处理 |

决策树、随机森林等树模型**不需要**特征缩放。

In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder, OneHotEncoder

sample = np.array([[1000, 0.01, 5], [2000, 0.02, 10], [500, 0.005, 3]])
print('原始数据（量纲差异巨大）：\n', sample)
print('\n标准化（StandardScaler）：\n', StandardScaler().fit_transform(sample).round(2))
print('\n归一化（MinMaxScaler）：\n', MinMaxScaler().fit_transform(sample).round(2))

### 7.2 类别特征编码

- **标签编码（LabelEncoder）**：适合有序类别或树模型
- **One-Hot 编码**：适合无序类别，避免引入虚假数值顺序

In [ ]:
categories = ['猫', '狗', '鸟', '狗', '猫']

le = LabelEncoder()
print('标签编码：', le.fit_transform(categories))
print('映射：', dict(zip(le.classes_, le.transform(le.classes_))))

ohe = OneHotEncoder(sparse_output=False)
encoded = ohe.fit_transform(np.array(categories).reshape(-1, 1))
df_ohe = pd.DataFrame(encoded.astype(int), columns=ohe.categories_[0])
df_ohe.insert(0, '原始', categories)
print('\nOne-Hot 编码：')
print(df_ohe.to_string(index=False))

### 7.3 处理缺失值

| 策略 | 方法 | 适用场景 |
|------|------|----------|
| 删除 | `dropna()` | 缺失比例极小 |
| 均值/中位数填充 | `SimpleImputer(strategy='mean')` | 数值型 |
| 众数填充 | `SimpleImputer(strategy='most_frequent')` | 类别型 |
| 模型预测填充 | `KNNImputer` | 缺失有规律可循时 |

In [ ]:
from sklearn.impute import SimpleImputer

data_nan = np.array([[1., 2., np.nan], [np.nan, 5., 6.], [7., np.nan, 9.]])
filled = SimpleImputer(strategy='mean').fit_transform(data_nan)

print('原始数据（含 NaN）：\n', data_nan)
print('\n均值填充后：\n', filled.round(2))

### 7.4 Pipeline（流水线）

将预处理和模型串联，**防止数据泄漏**（避免测试集信息流入训练阶段）。

In [ ]:
from sklearn.pipeline import Pipeline

pipeline = Pipeline([
    ('scaler',     StandardScaler()),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42))
])

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
pipeline.fit(X_tr, y_tr)
print(f'Pipeline 测试集准确率：{pipeline.score(X_te, y_te):.2%}')

---
## 第八章：超参数调优

- **网格搜索（GridSearchCV）**：穷举所有组合，精细但慢
- **随机搜索（RandomizedSearchCV）**：随机采样组合，快但可能错过最优

In [ ]:
from sklearn.model_selection import GridSearchCV

pipe = Pipeline([
    ('scaler',     StandardScaler()),
    ('classifier', RandomForestClassifier(random_state=42))
])

param_grid = {
    'classifier__n_estimators':   [50, 100, 200],
    'classifier__max_depth':      [None, 3, 5],
    'classifier__min_samples_split': [2, 5]
}

grid = GridSearchCV(pipe, param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid.fit(X_tr, y_tr)

print(f'最优参数：{grid.best_params_}')
print(f'CV 最优得分：{grid.best_score_:.2%}')
print(f'测试集得分：{grid.score(X_te, y_te):.2%}')

---
## 第九章：完整 ML 工作流实战

用糖尿病数据集走完完整流程：
**数据探索 → 预处理 → 训练多模型 → 评估 → 对比**

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error

diabetes = datasets.load_diabetes()
X_d, y_d = diabetes.data, diabetes.target

print(f'样本数：{len(X_d)}, 特征数：{X_d.shape[1]}')
print(f'目标变量范围：{y_d.min():.0f} ~ {y_d.max():.0f}')

fig, axes = plt.subplots(2, 5, figsize=(18, 7))
for i, feat in enumerate(diabetes.feature_names):
    ax = axes[i//5][i%5]
    ax.scatter(X_d[:, i], y_d, alpha=0.3, s=15, color='steelblue')
    ax.set_xlabel(feat)
    ax.set_ylabel('target')
plt.suptitle('各特征与目标变量的关系', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.ensemble import RandomForestRegressor

X_tr, X_te, y_tr, y_te = train_test_split(X_d, y_d, test_size=0.2, random_state=42)

reg_models = {
    '线性回归':   Pipeline([('s', StandardScaler()), ('m', LinearRegression())]),
    '随机森林回归': Pipeline([('s', StandardScaler()), ('m', RandomForestRegressor(n_estimators=100, random_state=42))]),
    '梯度提升回归': Pipeline([('s', StandardScaler()), ('m', GradientBoostingRegressor(n_estimators=100, random_state=42))]),
}

print(f'{"模型":<12} {"MAE":>8} {"RMSE":>8} {"R2":>8}')
print('-' * 40)
best_pipe, best_name, best_r2 = None, '', -np.inf
for name, pipe in reg_models.items():
    pipe.fit(X_tr, y_tr)
    pred = pipe.predict(X_te)
    mae  = mean_absolute_error(y_te, pred)
    rmse = np.sqrt(mean_squared_error(y_te, pred))
    r2   = r2_score(y_te, pred)
    print(f'{name:<12} {mae:>8.2f} {rmse:>8.2f} {r2:>8.3f}')
    if r2 > best_r2:
        best_r2, best_name, best_pipe = r2, name, pipe

print(f'\n最优模型：{best_name}（R2={best_r2:.3f}）')

In [ ]:
y_pred_best = best_pipe.predict(X_te)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].scatter(y_te, y_pred_best, alpha=0.6, color='steelblue')
axes[0].plot([y_te.min(), y_te.max()], [y_te.min(), y_te.max()], 'r--', lw=2)
axes[0].set_xlabel('真实值')
axes[0].set_ylabel('预测值')
axes[0].set_title(f'{best_name}：预测 vs 真实（R2={best_r2:.3f}）')

residuals = y_te - y_pred_best
axes[1].hist(residuals, bins=20, color='steelblue', edgecolor='white')
axes[1].axvline(0, color='r', linestyle='--')
axes[1].set_xlabel('残差')
axes[1].set_ylabel('频次')
axes[1].set_title(f'残差分布（均值 = {residuals.mean():.2f}）')

plt.tight_layout()
plt.show()

---
## 第十章：总结与学习建议

### 标准 ML 工作流

```
1. 问题定义
   监督/无监督？分类/回归？评估指标是什么？

2. 数据收集与探索（EDA）
   分布、相关性、缺失值、异常值

3. 特征工程
   缺失值处理 → 特征缩放 → 类别编码 → 特征选择

4. 模型选择与训练
   建立 Baseline → 尝试多个算法 → Pipeline

5. 模型评估
   交叉验证 → 混淆矩阵/ROC → 过拟合/欠拟合诊断

6. 超参数调优
   GridSearch / RandomizedSearch

7. 模型保存与部署
   joblib.dump → 预测服务
```

### 算法选择速查

| 场景 | 推荐起点 |
|------|----------|
| 分类，需要可解释性 | 逻辑回归、决策树 |
| 分类，追求性能 | 随机森林、梯度提升 |
| 分类，小数据集 | SVM |
| 回归 | 线性回归 → Ridge → 随机森林回归 |
| 聚类 | K-Means |
| 降维可视化 | PCA |

### 推荐后续方向

1. **进阶算法**：XGBoost / LightGBM、朴素贝叶斯
2. **实战练习**：Kaggle 竞赛入门赛题
3. **数学基础**：线性代数、概率论、微积分（梯度下降）
4. **推荐资源**：
   - scikit-learn 官方文档
   - 吴恩达《机器学习》课程
   - 《机器学习》周志华（西瓜书）

In [ ]:
import joblib

joblib.dump(best_pipe, '/tmp/best_model.pkl')
loaded = joblib.load('/tmp/best_model.pkl')
print('模型保存与加载成功')
print(f'预测结果一致：{np.allclose(loaded.predict(X_te), y_pred_best)}')